# Relay analysis

Reads every complete run folder under `results/<PLATFORM>/` and shows the metrics, the cold start breakdown and the charts.
The same code runs from the command line: `python3 analysis/make_charts.py`.

Set `PLATFORM = "synthetic"` to explore the fake data from `analysis/make_synthetic.py`. **Synthetic numbers are not results.**
Metric definitions are in the docstring of `analysis/relay_data.py`.

In [ ]:
PLATFORM = "gcp"   # or "synthetic"

import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() if Path("relay_data.py").exists() else Path.cwd() / "analysis"))
import relay_data as rd
import make_charts as mc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
runs = rd.load_runs(PLATFORM)
print(f"{len(runs)} complete runs")

## Per run metrics

In [ ]:
per_run = rd.metrics_table(runs)
per_run.sort_values(["study", "pattern", "strategy", "variant", "repeat"]) if not per_run.empty else per_run

## Summary: mean, min and max over repeats

In [ ]:
summary = rd.summary_table(per_run)
cols = ["study", "strategy", "pattern", "variant", "runs", "slo_violation_rate_mean", "p95_ttft_s_mean",
        "p95_latency_s_mean", "gpu_hours_mean", "time_to_scale_s_mean", "max_replicas_mean"]
summary[cols] if not summary.empty else summary

## Cold start breakdown, mean seconds per scale up

In [ ]:
cold = rd.coldstart_table(runs)
if cold.empty:
    print("no scale ups yet")
else:
    phases = ["decision_s", "node_start_s", "image_pull_s", "model_load_s", "other_s", "total_s"]
    display(cold.groupby(["study", "strategy", "new_node"])[phases].agg(["mean", "count"]).round(1))

## Charts

Regenerates every PNG, then shows them. Charts whose study has no runs yet are skipped.

In [ ]:
sys.argv = ["make_charts.py", "--platform", PLATFORM]
mc.main()
fig_dir = Path(mc.__file__).parent / "figures" / ("" if PLATFORM == "gcp" else PLATFORM)
for png in sorted(fig_dir.glob("*.png")):
    print(png.name)
    display(Image(filename=str(png), width=900))

## Questions to answer in the report

1. Spike: how long after the spike starts does each strategy add a replica, and how much of that is the decision versus the cold start?
2. Does Strategy B reach a lower SLO violation rate than A, and at how many GPU hours?
3. How much of the cold start does a warm spare remove, and what does the spare cost in GPU hours?
4. Which Strategy B threshold gives the best trade off between SLO violations and GPU hours?